# Лабораторна робота №2. Частина 2
**Тема:** Аналіз датасету Individual Household Electric Power Consumption.
Профілювання часу виконання здійснюється за допомогою модуля `timeit`.

In [1]:
import pandas as pd
import numpy as np
import timeit
import warnings
from sklearn.preprocessing import MinMaxScaler, StandardScaler
warnings.filterwarnings('ignore') 

## Завдання 1: Зчитування та Data Cleaning
Зчитати датасет. Видалити пропуски (у цьому датасеті вони позначені як `?`). 
Конвертувати стовпці у числові формати.

In [2]:
print("Починаю зчитування файлу...")
df_power = pd.read_csv('household_power_consumption.txt', sep=';', 
                       na_values=['?'], low_memory=False)

print(f"Дані зчитано. Початковий розмір: {df_power.shape}")

df_power = df_power.dropna()
print(f"Розмір після видалення пропусків: {df_power.shape}")

columns_to_convert = ['Global_active_power', 'Global_reactive_power', 'Voltage', 
                      'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']

for col in columns_to_convert:
    df_power[col] = pd.to_numeric(df_power[col])

print("Data Cleaning завершено успішно!")
display(df_power.head())

Починаю зчитування файлу...
Дані зчитано. Початковий розмір: (2075259, 9)
Розмір після видалення пропусків: (2049280, 9)
Data Cleaning завершено успішно!


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


## Завдання 2.1: Вибірка за потужністю
Обрати всі записи, у яких загальна активна споживана потужність (`Global_active_power`) перевищує 5 кВт.

In [3]:
def filter_high_power(df):
    return df[df['Global_active_power'] > 5.0]

time_taken = timeit.timeit(lambda: filter_high_power(df_power), number=1)
result_1 = filter_high_power(df_power)

print(f"Час виконання: {time_taken:.5f} секунд")
print(f"Знайдено записів: {len(result_1)}")
display(result_1.head())

Час виконання: 0.00517 секунд
Знайдено записів: 17547


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
1,16/12/2006,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
11,16/12/2006,17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0
12,16/12/2006,17:36:00,5.224,0.478,232.99,22.4,0.0,1.0,16.0


## Завдання 2.2: Вибірка за силою струму та споживанням
Обрати всі записи, у яких сила струму лежить в межах 19-20 А (`Global_intensity`), для них виявити ті, у яких пральна машина та холодильник (`Sub_metering_2`) споживають більше, ніж бойлер та кондиціонер (`Sub_metering_3`).

In [4]:
def filter_appliances(df):
    return df[(df['Global_intensity'] >= 19) & 
              (df['Global_intensity'] <= 20) & 
              (df['Sub_metering_2'] > df['Sub_metering_3'])]

time_taken = timeit.timeit(lambda: filter_appliances(df_power), number=1)
result_2 = filter_appliances(df_power)

print(f"Час виконання: {time_taken:.5f} секунд")
print(f"Знайдено записів: {len(result_2)}")
display(result_2.head())

Час виконання: 0.01137 секунд
Знайдено записів: 2509


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
45,16/12/2006,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0
460,17/12/2006,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0
464,17/12/2006,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0
475,17/12/2006,01:19:00,4.636,0.140,237.37,19.4,0.0,36.0,0.0
476,17/12/2006,01:20:00,4.634,0.152,237.17,19.4,0.0,35.0,0.0


## Завдання 2.3: Випадкова вибірка та середні величини
Обрати випадковим чином 500 000 записів (без повторів). Обчислити середні величини усіх 3-х груп споживання (`Sub_metering_1`, `Sub_metering_2`, `Sub_metering_3`).

In [5]:
def random_sample_means(df):
    sample_df = df.sample(n=500000, replace=False, random_state=42)
    
    means = sample_df[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()
    return means

time_taken = timeit.timeit(lambda: random_sample_means(df_power), number=1)
result_3 = random_sample_means(df_power)

print(f"Час виконання: {time_taken:.5f} секунд")
print("Середні величини груп споживання (Вт-год):")
print(result_3)

Час виконання: 0.12243 секунд
Середні величини груп споживання (Вт-год):
Sub_metering_1    1.119258
Sub_metering_2    1.308912
Sub_metering_3    6.452950
dtype: float64


## Завдання 2.4: Складна вибірка (вечір, потужність, групи)
Записи після 18:00 із потужністю > 6 кВт. Група 2 (пралка/холодильник) має бути найбільшою. З першої половини відібраних беремо кожен 3-й результат, з другої - кожен 4-й.

In [6]:
def filter_complex_evening(df):
    cond1 = df['Time'] >= '18:00:00'
    cond2 = df['Global_active_power'] > 6.0
    
    cond3 = (df['Sub_metering_2'] > df['Sub_metering_1']) & (df['Sub_metering_2'] > df['Sub_metering_3'])
    
    filtered_df = df[cond1 & cond2 & cond3]
    
    half_idx = len(filtered_df) // 2
    first_half = filtered_df.iloc[:half_idx]
    second_half = filtered_df.iloc[half_idx:]
    
    res1 = first_half.iloc[::3]
    res2 = second_half.iloc[::4]
    
    return pd.concat([res1, res2])

time_taken = timeit.timeit(lambda: filter_complex_evening(df_power), number=1)
result_4 = filter_complex_evening(df_power)

print(f"Час виконання: {time_taken:.5f} секунд")
print(f"Відібрано записів після всіх фільтрів: {len(result_4)}")
display(result_4.head())

Час виконання: 0.08296 секунд
Відібрано записів після всіх фільтрів: 310


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
41,16/12/2006,18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0
44,16/12/2006,18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0
17494,28/12/2006,20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0
17498,28/12/2006,21:02:00,8.088,0.262,235.50,34.4,1.0,72.0,17.0
17501,28/12/2006,21:05:00,7.230,0.152,235.22,30.6,1.0,73.0,17.0


## Завдання 3: Нормування та стандартизація
Пронормувати (Min-Max) та стандартизувати (Z-score) числові атрибути датасету.

In [7]:
def apply_scaling(df, columns):
    min_max_scaler = MinMaxScaler()
    z_score_scaler = StandardScaler()
    
    norm_data = pd.DataFrame(min_max_scaler.fit_transform(df[columns]), columns=columns)
    std_data = pd.DataFrame(z_score_scaler.fit_transform(df[columns]), columns=columns)
    
    return norm_data, std_data

target_columns = ['Global_active_power', 'Voltage', 'Global_intensity']

start_time = timeit.default_timer()
df_normalized, df_standardized = apply_scaling(df_power, target_columns)
end_time = timeit.default_timer()

print(f" Час виконання перетворень: {end_time - start_time:.5f} секунд")

print("\n Результат Min-Max нормалізації (значення від 0 до 1)")
display(df_normalized.head(3))

print("\n Результат Z-score стандартизації (середнє 0, відхилення 1)")
display(df_standardized.head(3))

 Час виконання перетворень: 0.17078 секунд

 Результат Min-Max нормалізації (значення від 0 до 1)


,Global_active_power,Voltage,Global_intensity
0,0.374796,0.376090,0.377593
1,0.478363,0.336995,0.473029
2,0.479631,0.326010,0.473029



 Результат Z-score стандартизації (середнє 0, відхилення 1)


,Global_active_power,Voltage,Global_intensity
0,2.955077,-1.851816,3.098789
1,4.037085,-2.225274,4.133800
2,4.050326,-2.330213,4.133800


## Завдання 4: Кореляція
Підрахувати коефіцієнт Пірсона та Спірмена для двох real атрибутів (`Global_active_power` та `Global_intensity`).

In [9]:
import timeit

def task_correlation_realistic(df):
    col1, col2 = 'Global_active_power', 'Global_reactive_power'
    
    pearson_corr = df[col1].corr(df[col2], method='pearson')
    
    spearman_corr = df[col1].corr(df[col2], method='spearman')
    
    return col1, col2, pearson_corr, spearman_corr

start_time = timeit.default_timer()
c1, c2, p_val, s_val = task_correlation_realistic(df_power) 
end_time = timeit.default_timer()

print(f"Час виконання: {end_time - start_time:.5f} секунд\n")
print(f"Кореляція між {c1} та {c2}:")
print(f" - Коефіцієнт Пірсона:  {p_val:.4f}")
print(f" - Коефіцієнт Спірмена: {s_val:.4f}")

Час виконання: 0.40296 секунд

Кореляція між Global_active_power та Global_reactive_power:
 - Коефіцієнт Пірсона:  0.2470
 - Коефіцієнт Спірмена: 0.2693


## Завдання 5: One Hot Encoding
Провести One Hot Encoding категоріального атрибута. Створимо категоріальний атрибут "Місяць" із дати і закодуємо його.

In [47]:
def apply_one_hot_encoding(df):
    months = df['Date'].str.split('/').str[1]
    return pd.get_dummies(months, prefix='Month', dtype=int)

start_time = timeit.default_timer()
ohe_result = apply_one_hot_encoding(df_power)
end_time = timeit.default_timer()

print(f" Час виконання One Hot Encoding: {end_time - start_time:.5f} секунд")
print("Результат кодування місяців (перші 3 рядки):")
display(ohe_result.head(3))

 Час виконання One Hot Encoding: 1.61853 секунд
Результат кодування місяців (перші 3 рядки):


,Month_1,Month_10,Month_11,Month_12,Month_2,Month_3,Month_4,Month_5,Month_6,Month_7,Month_8,Month_9
0,0,0,0,1,0,0,0,0,0,0,0,0
1,0,0,0,1,0,0,0,0,0,0,0,0
2,0,0,0,1,0,0,0,0,0,0,0,0
